EXPERIMENT NO. 9 (Group B) 
 
Aim: Data Wrangling 
Problem Statement: Data Wrangling on Real Estate Market 
Dataset: "RealEstate_Prices.csv" 
Description: The dataset contains information about housing prices in a specific real estate market. It 
includes various attributes such as property characteristics, location, sale prices, and other relevant 
features. The goal is to perform data wrangling to gain insights into the factors influencing housing prices 
and prepare the dataset for further analysis or modeling.


Name:Aditya Rade
Roll.No:BEAD23167

Outputs:


Step 1 — Import the libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams["figure.figsize"] = (20, 10)

Step 2 — Load and inspect the dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("Real estate.csv")

print(df.head())
print("\nShape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())

   No  X1 transaction date  X2 house age  \
0   1             2012.917          32.0   
1   2             2012.917          19.5   
2   3             2013.583          13.3   
3   4             2013.500          13.3   
4   5             2012.833           5.0   

   X3 distance to the nearest MRT station  X4 number of convenience stores  \
0                                84.87882                               10   
1                               306.59470                                9   
2                               561.98450                                5   
3                               561.98450                                5   
4                               390.56840                                5   

   X5 latitude  X6 longitude  Y house price of unit area  
0     24.98298     121.54024                        37.9  
1     24.98034     121.53951                        42.2  
2     24.98746     121.54391                        47.3  
3     24.98746     121.54391  

Step 2 — Clean column names + missing values

In [3]:
# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
)

print("Cleaned columns:")
print(df.columns)

# Remove duplicate rows
df = df.drop_duplicates()

# Handle missing values
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

print("\nMissing values after cleaning:")
print(df.isnull().sum())

Cleaned columns:
Index(['no', 'x1_transaction_date', 'x2_house_age',
       'x3_distance_to_the_nearest_mrt_station',
       'x4_number_of_convenience_stores', 'x5_latitude', 'x6_longitude',
       'y_house_price_of_unit_area'],
      dtype='object')

Missing values after cleaning:
no                                        0
x1_transaction_date                       0
x2_house_age                              0
x3_distance_to_the_nearest_mrt_station    0
x4_number_of_convenience_stores           0
x5_latitude                               0
x6_longitude                              0
y_house_price_of_unit_area                0
dtype: int64


Step 3 — Explore, filter and aggregate

In [4]:
print("\nDataset information:")
df.info()

print("\nStatistical summary:")
print(df.describe())

# Display categorical columns
print("\nCategorical columns:")
print(df.select_dtypes(include="object").columns)

# Example filtering
# Change the column name below if your dataset uses a different one
for col in df.select_dtypes(include="object").columns:
    print(f"\nValues in {col}:")
    print(df[col].value_counts().head())

# Numerical correlation
print("\nCorrelation:")
print(df.select_dtypes(include=np.number).corr())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 8 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   no                                      414 non-null    int64  
 1   x1_transaction_date                     414 non-null    float64
 2   x2_house_age                            414 non-null    float64
 3   x3_distance_to_the_nearest_mrt_station  414 non-null    float64
 4   x4_number_of_convenience_stores         414 non-null    int64  
 5   x5_latitude                             414 non-null    float64
 6   x6_longitude                            414 non-null    float64
 7   y_house_price_of_unit_area              414 non-null    float64
dtypes: float64(6), int64(2)
memory usage: 26.0 KB

Statistical summary:
               no  x1_transaction_date  x2_house_age  \
count  414.000000           414.000000    414.000000   

Step 4 — Encode categorical variables

In [5]:
# One-hot encoding of categorical variables
categorical_cols = df.select_dtypes(include="object").columns

df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

print("Encoded dataset:")
print(df_encoded.head())

print("\nNew shape:", df_encoded.shape)

Encoded dataset:
   no  x1_transaction_date  x2_house_age  \
0   1             2012.917          32.0   
1   2             2012.917          19.5   
2   3             2013.583          13.3   
3   4             2013.500          13.3   
4   5             2012.833           5.0   

   x3_distance_to_the_nearest_mrt_station  x4_number_of_convenience_stores  \
0                                84.87882                               10   
1                               306.59470                                9   
2                               561.98450                                5   
3                               561.98450                                5   
4                               390.56840                                5   

   x5_latitude  x6_longitude  y_house_price_of_unit_area  
0     24.98298     121.54024                        37.9  
1     24.98034     121.53951                        42.2  
2     24.98746     121.54391                        47.3  
3     24.9874

Step 5 — Detect and remove numerical outliers

In [6]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("Shape after outlier removal:", df.shape)

Shape after outlier removal: (303, 8)


Step 6 — Final cleaned dataset

In [8]:
print("Final dataset:")
print(df.head())

print("\nFinal shape:", df.shape)

print("\nFinal missing values:")
print(df.isnull().sum())

# Save cleaned dataset
df.to_csv("Real_estate_cleaned.csv", index=False)

print("\nCleaned dataset saved successfully!")
print("Name: Aditya Rade Roll No: BEAD23167")

Final dataset:
   no  x1_transaction_date  x2_house_age  \
0   1             2012.917          32.0   
1   2             2012.917          19.5   
2   3             2013.583          13.3   
3   4             2013.500          13.3   
4   5             2012.833           5.0   

   x3_distance_to_the_nearest_mrt_station  x4_number_of_convenience_stores  \
0                                84.87882                               10   
1                               306.59470                                9   
2                               561.98450                                5   
3                               561.98450                                5   
4                               390.56840                                5   

   x5_latitude  x6_longitude  y_house_price_of_unit_area  
0     24.98298     121.54024                        37.9  
1     24.98034     121.53951                        42.2  
2     24.98746     121.54391                        47.3  
3     24.98746 